# 1) Load Data
Loading raw game data collected from the Steam API. Data contains nested JSON fields that need to be flattened before analysis.

In [ ]:
import pandas as pd
import ast

In [ ]:
data = pd.read_csv("../data/raw/games_raw.csv")

print(data.shape)
print(data.isnull().sum())
display(data.dtypes)

In [ ]:
data.head()

# 2) Pre-processing Steps
Extracting relevant values from nested dictionary columns, converting price to USD, standardizing date format, and dropping columns that are non-relavant to the goal.

## 2.1) Parse Nested Columns

In [ ]:
data["recommendations"] = data["recommendations"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["recommendations"] = data["recommendations"].apply(lambda x: x.get("total") if not pd.isna(x) else None)

data["metacritic"] = data["metacritic"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["metacritic"] = data["metacritic"].apply(lambda x: x.get("score") if not pd.isna(x) else None)

## 2.2) Clean Developers
Converted developers column from a list of names to a count as the number of developers servers as a proxy for studio size, distinguishing indie solo or small team projects from large studio productions. The raw developer names are perseved in the games_raw.csv if needed for future analysis.

In [ ]:
data["developers"] = data["developers"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["developers"] = data["developers"].apply(lambda x: len(x) if x is not None else 0)

## 2.3) Clean Genres
15 games with non-English genre tags are excluded from genre analysis which is less than 0.1% of the dataset.

In [ ]:
data["genres"] = data["genres"].str.replace("Free-to-play", "Free To Play")
data["genres"] = data["genres"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["genres"] = data["genres"].apply(lambda x: [desc.get("description") for desc in x] if x is not None else None)

In [ ]:
NON_ENGLISH = ["Aventura", "独立", "竞速", "模拟", "动作", "冒险", "角色扮演", "抢先体验", "Akční", "Strategické", "Aksiyon", "Macera", "RYO", "Azione", "Avventura", "Казуальные игры", 
               "Инди", "Indépendant", "Stratégie", "Aventure", "Abenteuer", "Simuladores", "Actie", "Eventyr", "休闲", "策略"]

display(data["genres"].explode().unique())

count = data["genres"].apply(lambda x: any(i in NON_ENGLISH for i in x) if x is not None else False).sum()
print(f"Games with non-English genres: {count}")

In [ ]:
VALID_GENRES = ["Action", "Strategy", "Adventure", "Indie", "Simulation", "RPG", "Casual", "Racing", "Massively Multiplayer", "Sports", "Early Access"]

data["genres"] = data["genres"].apply(lambda x: [valid for valid in x if valid in VALID_GENRES] if x is not None else None)

## 2.4) Clean Price

In [ ]:
data["price_overview"] = data["price_overview"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["price_overview"] = (data["price_overview"].apply(lambda x: 0 if pd.isna(x) else x.get("final") if x.get("currency") == "USD" else None) / 100)

## 2.5) Format Dates

In [ ]:
data["release_date"] = data["release_date"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["release_date"] = data["release_date"].apply(lambda x: x.get("date") if not pd.isna(x) else None)
data["release_date"] = pd.to_datetime(data["release_date"], format="mixed", errors="coerce")

## 2.6) Configure Columns and Rows

Rows missing values in developers, publishers, price_usd, genres, and release_date were dropped as these fields are central to the goal, which reduced the data from 22,171 to 20,516 rows. Metacritic scores were retained as null rather than dropped as that would reduce the size of the data significantly and impact future analysis. Dropping these would introduce significant survivorship bias toward larger, more established titles. 17,065 games have no recommendation data, which was confirmed against Steam store pages. These are retained in the dataset but will be excluded from analyses requiring review metrics.

In [ ]:
data = data.rename(columns={"price_overview": "price_usd", "developers": "developer_count"})
data = data.drop(columns=["is_free", "categories", "publishers"])
data = data.dropna(subset=["price_usd", "genres", "release_date"])

In [ ]:
print(data.shape)
print(data.isnull().sum())
display(data.dtypes)

In [ ]:
data.head()

# 3. Save Data

In [ ]:
data.to_csv("../data/processed/games_clean.csv", index=False)